# Ch. 4 · TP1 Du neurone au MLP

[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/benlahmar/IASciences/blob/main/notebooks/ch04-reseaux-neurones.ipynb)

**IA pour les sciences — M1.** Programmer un neurone à la main, comprendre la rétropropagation, puis comparer un MLP à un modèle simple.

**Durée indicative :** 1 h 30 (partie A du TP de 3 h)

**Objectifs**

- Implémenter en NumPy un neurone sigmoïde entraîné par descente de gradient.
- Vérifier un gradient calculé par la règle de chaîne avec une différence finie.
- Montrer avec le problème XOR pourquoi les couches cachées non linéaires sont nécessaires.
- Comparer un MLP scikit-learn à une régression logistique avec un protocole sans fuite.

**Prérequis :** chapitre 3 ; dérivée d’une fonction composée.

> **Mode d’emploi.** Exécutez les cellules dans l’ordre. Les cellules marquées **✏️ Exercice** contiennent des `...` à compléter : une cellule non complétée lève une erreur, c’est normal. Les corrigés sont repliés sous **« Voir le corrigé »** : cherchez d’abord ! Les cellules suivantes ne dépendent jamais du résultat d’un exercice.

## 1. Un neurone, écrit à la main

Pour une entrée $x\in\mathbb{R}^p$ : $z = w^\top x + b$, $\hat p = \sigma(z) = 1/(1+e^{-z})$.
La perte d’entropie croisée binaire est $\mathcal{L} = -\frac1n\sum [y\log\hat p + (1-y)\log(1-\hat p)]$.

**Rétropropagation (règle de chaîne).** Comme $\sigma'(z)=\sigma(z)(1-\sigma(z))$, on obtient le résultat remarquablement simple :

$$\frac{\partial \mathcal{L}}{\partial z_i} = \frac{1}{n}(\hat p_i - y_i), \qquad
\frac{\partial \mathcal{L}}{\partial w} = \frac1n X^\top(\hat p - y), \qquad
\frac{\partial \mathcal{L}}{\partial b} = \frac1n\sum_i(\hat p_i - y_i).$$

La descente de gradient met à jour $w \leftarrow w - \eta\,\partial\mathcal{L}/\partial w$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
n = 200
X = rng.normal(size=(n, 2))
y = (X[:, 0] + 0.5 * X[:, 1] + rng.normal(0, 0.5, n) > 0).astype(float)

def sigmoide(z):
    return 1 / (1 + np.exp(-z))

def perte_et_gradients(w, b, X, y):
    p = sigmoide(X @ w + b)
    eps = 1e-12
    perte = -np.mean(y * np.log(p + eps) + (1 - y) * np.log(1 - p + eps))
    dz = (p - y) / len(y)
    return perte, X.T @ dz, dz.sum()

w, b, eta = np.zeros(2), 0.0, 0.5
historique = []
for epoque in range(200):
    perte, gw, gb = perte_et_gradients(w, b, X, y)
    w, b = w - eta * gw, b - eta * gb
    historique.append(perte)
print('Poids appris :', w.round(3), '— biais :', round(b, 3))
print('Exactitude d’entraînement :', np.mean((sigmoide(X @ w + b) > .5) == y))
plt.plot(historique); plt.xlabel('Époque'); plt.ylabel('Entropie croisée'); plt.title('Apprentissage d’un neurone'); plt.show()

**🔎 À observer.** Le rapport $w_2/w_1$ appris est proche de 0,5 : le neurone a retrouvé la direction de la frontière simulée.

## 2. Démonstration : vérifier un gradient

Toute implémentation de gradient se vérifie par **différences finies** : $\partial\mathcal{L}/\partial w_j \approx
[\mathcal{L}(w+h e_j)-\mathcal{L}(w-h e_j)]/2h$. C’est ce que font les bibliothèques lors de leurs tests.

In [ ]:
w0, b0, h = np.array([0.3, -0.2]), 0.1, 1e-5
_, g_analytique, _ = perte_et_gradients(w0, b0, X, y)
g_numerique = np.zeros(2)
for j in range(2):
    e = np.zeros(2); e[j] = h
    g_numerique[j] = (perte_et_gradients(w0 + e, b0, X, y)[0] - perte_et_gradients(w0 - e, b0, X, y)[0]) / (2 * h)
print('Gradient analytique :', g_analytique.round(6))
print('Gradient numérique  :', g_numerique.round(6))

### ✏️ Exercice 1 — Influence du taux d’apprentissage

Relancez l’apprentissage avec $\eta \in \{0{,}01;\ 0{,}5;\ 5;\ 50\}$ et tracez les quatre courbes de perte sur un même graphique (échelle log). Que se passe-t-il pour une valeur trop grande ?

In [ ]:
for eta in [0.01, 0.5, 5, 50]:
    w, b, hist = np.zeros(2), 0.0, []
    for epoque in range(200):
        perte, gw, gb = ...
        w, b = ..., ...
        hist.append(perte)
    plt.plot(hist, label=f'eta = {eta}')
plt.yscale('log'); plt.legend(); plt.xlabel('Époque'); plt.show()

<details><summary><b>Voir le corrigé</b></summary>

```python
for eta in [0.01, 0.5, 5, 50]:
    w, b, hist = np.zeros(2), 0.0, []
    for epoque in range(200):
        perte, gw, gb = perte_et_gradients(w, b, X, y)
        w, b = w - eta * gw, b - eta * gb
        hist.append(perte)
    plt.plot(hist, label=f'eta = {eta}')
plt.yscale('log'); plt.legend(); plt.xlabel('Époque'); plt.show()
```

Avec $\eta=0{,}01$, la perte baisse très lentement. Avec un $\eta$ très grand, les pas sont énormes : la perte oscille ou stagne à un niveau élevé (les sigmoïdes saturent). Le taux d’apprentissage est l’hyperparamètre le plus important d’un réseau.

</details>

## 3. Démonstration : XOR, ou pourquoi il faut des couches cachées

Le problème XOR (classe 1 si exactement une des deux coordonnées est positive) n’est **pas linéairement séparable** :
aucune droite ne sépare les classes. Un modèle linéaire plafonne vers 50 % ; un MLP avec une couche cachée ReLU réussit.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier

Xx = rng.uniform(-1, 1, size=(400, 2))
yx = ((Xx[:, 0] > 0) ^ (Xx[:, 1] > 0)).astype(int)
lin = LogisticRegression().fit(Xx, yx)
mlp_xor = MLPClassifier(hidden_layer_sizes=(8,), activation='relu', max_iter=3000, random_state=0).fit(Xx, yx)

gx, gy = np.meshgrid(np.linspace(-1, 1, 200), np.linspace(-1, 1, 200))
grille = np.c_[gx.ravel(), gy.ravel()]
fig, axes = plt.subplots(1, 2, figsize=(10, 4.3))
for ax, (nom, m) in zip(axes, [('Régression logistique', lin), ('MLP (8 neurones ReLU)', mlp_xor)]):
    ax.contourf(gx, gy, m.predict(grille).reshape(gx.shape), alpha=.25, cmap='coolwarm')
    ax.scatter(Xx[:, 0], Xx[:, 1], c=yx, cmap='coolwarm', s=10)
    ax.set_title(f'{nom} — exactitude {m.score(Xx, yx):.2f}')
plt.tight_layout(); plt.show()

## 4. MLP sur des données réelles : faire mieux qu’une baseline ?

Retour au jeu Iris (4 mesures, 3 espèces). Le jeu est **petit** : c’est l’occasion de constater qu’un réseau n’est pas
automatiquement meilleur qu’un modèle linéaire.

In [ ]:
import warnings
import pandas as pd
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings('ignore', category=ConvergenceWarning)
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report

iris = load_iris(as_frame=True)
Xi, yi = iris.data, iris.target
X_train, X_test, y_train, y_test = train_test_split(Xi, yi, test_size=0.25, stratify=yi, random_state=42)

baseline = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(X_train, y_train)
print('Logistique — exactitude test :', round(accuracy_score(y_test, baseline.predict(X_test)), 3))

mlp = make_pipeline(StandardScaler(),
                    MLPClassifier(hidden_layer_sizes=(16, 8), activation='relu', solver='adam',
                                  alpha=0.01, max_iter=3000, random_state=42))
mlp.fit(X_train, y_train)
print('MLP        — exactitude test :', round(accuracy_score(y_test, mlp.predict(X_test)), 3))
print(classification_report(y_test, mlp.predict(X_test), target_names=iris.target_names))

reseau = mlp.named_steps['mlpclassifier']
plt.plot(reseau.loss_curve_)
plt.xlabel('Itération'); plt.ylabel('Perte d’entraînement'); plt.title('Courbe d’apprentissage du MLP'); plt.show()
n_params = sum(c.size for c in reseau.coefs_) + sum(b.size for b in reseau.intercepts_)
print('Nombre de paramètres du MLP :', n_params, '— pour', len(X_train), 'fleurs d’entraînement')

**Démonstration : l’early stopping sur un très petit jeu.** `early_stopping=True` réserve 20 % de l’entraînement (ici ~22 fleurs)
pour décider quand s’arrêter. Avec si peu d’observations, le critère est bruité : l’arrêt peut être prématuré.

In [ ]:
for patience in [10, 50, 200]:
    m = make_pipeline(StandardScaler(),
                      MLPClassifier(hidden_layer_sizes=(16, 8), alpha=0.01, max_iter=3000, early_stopping=True,
                                    validation_fraction=0.2, n_iter_no_change=patience, random_state=42)).fit(X_train, y_train)
    print(f'patience {patience:3d} : arrêt après {m[-1].n_iter_:4d} itérations, exactitude test {m.score(X_test, y_test):.3f}')

**🔎 À retenir.** L’early stopping est précieux avec beaucoup de données (TP2, PyTorch) ; sur 112 observations, la validation interne
est trop petite pour être un juge fiable. Le résultat dépend alors d’un hyperparamètre de plus.

## 5. Choisir l’architecture par validation croisée (jamais sur le test)

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
for layers in [(4,), (8,), (16, 8), (64, 64)]:
    pipe = make_pipeline(StandardScaler(),
                         MLPClassifier(hidden_layer_sizes=layers, max_iter=2000, alpha=0.01, random_state=42))
    scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring='accuracy')
    print(f'{str(layers):10s} exactitude CV : {scores.mean():.3f} ± {scores.std():.3f}')

### ✏️ Exercice 2 — Régularisation L2

Pour l’architecture `(64, 64)`, faites varier `alpha` dans `[1e-4, 1e-2, 1, 10, 30, 100]` et affichez l’exactitude CV. Que fait une régularisation trop forte ?

In [ ]:
for alpha in [1e-4, 1e-2, 1, 10, 30, 100]:
    pipe = make_pipeline(StandardScaler(),
                         MLPClassifier(hidden_layer_sizes=(64, 64), alpha=..., max_iter=3000, random_state=42))
    scores = ...
    print(alpha, round(scores.mean(), 3), '±', round(scores.std(), 3))

<details><summary><b>Voir le corrigé</b></summary>

```python
for alpha in [1e-4, 1e-2, 1, 10, 30, 100]:
    pipe = make_pipeline(StandardScaler(),
                         MLPClassifier(hidden_layer_sizes=(64, 64), alpha=alpha, max_iter=3000, random_state=42))
    scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring='accuracy')
    print(alpha, round(scores.mean(), 3), '±', round(scores.std(), 3))
```

`alpha` ajoute $\alpha\|W\|^2$ à la perte. Jusqu’à `alpha=10`, rien ne change : sur Iris, le réseau ne surapprend pas. Au-delà, les poids sont écrasés : le réseau devient presque constant (sous-apprentissage) et l’exactitude tombe vers le hasard (1/3).

</details>

## Interprétation scientifique (à rédiger)

1. Le MLP améliore-t-il réellement la régression logistique ? Les écarts sont-ils supérieurs à la variabilité de la validation croisée ?
2. Combien de paramètres le MLP possède-t-il par rapport au nombre d’observations ? Quelles conséquences ?
3. Qu’a montré l’exemple XOR sur le rôle des activations non linéaires ?
4. Quels jeux indépendants faudrait-il pour valider une identification botanique sur de nouvelles régions ou appareils ?